# V0 spatial-residual DNN: orthogonal residuals, pad-plane boundary, and parallel side training

This notebook improves the first full spatial-residual model in four important ways.

1. **The original `residual_r` is not used as the radial target.** The fit point was evaluated at the measured cluster radius, so that branch is artificially concentrated at zero. Instead, the notebook constructs the measured-minus-fit displacement in the transverse plane, projects it onto the fitted-track normal, and trains the detector field through

\[
d_n^{\rm pred}
=
\Delta r\,(\hat n\!\cdot\!\hat e_r)
+
r\Delta\phi\,(\hat n\!\cdot\!\hat e_\phi).
\]

Tracks crossing the same detector region at different angles separate the underlying \(\Delta r\) and \(r\Delta\phi\) components.

2. **Each side is evaluated only in its physical z domain.**

```text
side 0: -z_pad <= z <= 0
side 1:       0 <= z <= +z_pad
```

The two sides are allowed to differ at the central membrane.

3. **Every correction is exactly zero at its own pad plane.** The network output is multiplied by

\[
u = 1-|z|/z_{\rm pad},
\]

so the correction can be sizable at \(|z|=90\) cm but must become exactly zero at \(|z|=z_{\rm pad}\).

4. **North and South train simultaneously.** Two independent side workers run in parallel. Each worker receives several PyTorch/BLAS threads, while residual pretraining uses large flattened-cluster batches.

The main switch remains:

```python
include_z_correction = False
```

Set it to `True` to train \(\Delta z\) together with the transverse field.


In [ ]:
# ============================================================
# Imports and configuration
# ============================================================

from pathlib import Path
from array import array
import copy
import json
import math
import os
import random
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from scipy.optimize import minimize

try:
    from joblib import Parallel, delayed, parallel_backend
    joblib_available = True
except ImportError:
    joblib_available = False
    print('WARNING: joblib is unavailable; the two sides will train sequentially.')

try:
    import ROOT as root
except ImportError:
    root = None

seed = 12345
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

torch.set_float32_matmul_precision('high')

candidate_csv = Path('/media/yoren/T7_Shield/sPHENIX/tracking/csvs/fit_k0s_pp_new_v0_candidates.csv')
cluster_csv = Path('/media/yoren/T7_Shield/sPHENIX/tracking/csvs/fit_k0s_pp_new_v0_clusters.csv')

work_dir = Path('output/v0_spatial_residual_dnn_v2')
work_dir.mkdir(parents=True, exist_ok=True)

checkpoint_dir = work_dir / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

cache_dir = work_dir / 'tensor_cache'
cache_dir.mkdir(parents=True, exist_ok=True)

# Main physics switch.
include_z_correction = False

training_parity = 0
validation_parity = 1 - training_parity

pdg_kshort_mass = 0.497611
pion_mass = 0.13957039
proton_mass = 0.9382720813

magnetic_field_tesla = 1.4
curvature_conversion = 0.003

# Physical TPC half-length / pad-plane location.
z_pad_cm = 102.325
central_membrane_tolerance_cm = 0.50
pad_plane_tolerance_cm = 0.75

maximum_clusters = 48
minimum_clusters_per_track = 20
use_recommended_clusters_only = True
use_kalman_used_clusters_only = False

# Outlier rejection for the physically constrained residual targets.
maximum_abs_residual_normal_cm = 1.0
maximum_abs_residual_z_cm = 2.0

# Fallback measurement uncertainties.
fallback_sigma_r_cm = 0.20
fallback_sigma_rphi_cm = 0.05
fallback_sigma_z_cm = 0.10
minimum_sigma_normal_cm = 0.02

# Signed output bounds. The hard pad-plane factor multiplies these values.
maximum_delta_r_cm = 0.50
maximum_r_delta_phi_cm = 0.30
maximum_delta_z_cm = 0.50

# Smooth detector-space representation.
number_of_radial_basis_functions = 12
radial_basis_minimum_cm = 31.0
radial_basis_maximum_cm = 76.0
radial_basis_width_scale = 1.35
number_of_phi_harmonics = 3

hidden_width = 32#64
hidden_layers = 2#3
dropout_probability = 0.03

learning_rate = 1.0e-3
weight_decay = 1.0e-3

# Large cluster-only batches make residual pretraining much faster.
residual_batch_size = 32768
candidate_batch_size = 1024
validation_batch_size = 2048

residual_pretraining_epochs = 100
hybrid_training_epochs = 100
validation_interval = 5
patience = 40

# Hybrid-loss coefficients.
mass_loss_weight = 1.0
residual_loss_weight = 0.15
smoothness_loss_weight = 0.05
amplitude_loss_weight = 0.01
mean_field_loss_weight = 0.002

# Smooth daughter-pT reweighting.
use_smooth_track_pt_reweighting = True
track_pt_weight_power = 0.50
minimum_pair_weight = 0.35
maximum_pair_weight = 3.00

# Two ensemble rounds. In every round, side 0 and side 1 train concurrently.
number_of_ensemble_models = 1#2

# CPU parallelism. Override requested_total_cpu_threads to match the allocation.
requested_total_cpu_threads = 12
available_cpu_threads = int(
    os.environ.get(
        'SLURM_CPUS_PER_TASK',
        os.cpu_count() or requested_total_cpu_threads,
    )
)
total_cpu_threads = max(
    2,
    min(requested_total_cpu_threads, available_cpu_threads),
)
parallel_side_jobs = 2
threads_per_side = max(1, total_cpu_threads // parallel_side_jobs)

kshort_pt_bin_edges = np.array([
    0.5, 0.8, 1.1, 1.4, 1.8, 2.2, 3.0,
], dtype=np.float64)

print('PyTorch:', torch.__version__)
print('include_z_correction:', include_z_correction)
print('available_cpu_threads:', available_cpu_threads)
print('total_cpu_threads:', total_cpu_threads)
print('parallel_side_jobs:', parallel_side_jobs)
print('threads_per_side:', threads_per_side)


## Load the linked tables and construct the transverse normal residual

The cluster CSV already contains measured positions, fitted positions, and fitted local momenta. No new exporter columns are required.

For each cluster, define the fitted transverse tangent and one normal direction:

\[
\hat t_T = \frac{(p_x^{\rm fit},p_y^{\rm fit})}{p_T^{\rm fit}},
\qquad
\hat n_T=(-t_y,t_x).
\]

The measured-minus-fit displacement is

\[
\mathbf d_T=(x_{\rm cluster}-x_{\rm fit},y_{\rm cluster}-y_{\rm fit}).
\]

The usable transverse observation is

\[
d_n=\mathbf d_T\cdot\hat n_T.
\]

The sign choice of \(\hat n_T\) is irrelevant because the target and the field-projection coefficients change sign together.


In [ ]:
# ============================================================
# Load tables and derive orthogonal transverse residual quantities
# ============================================================

candidates = pd.read_csv(candidate_csv)
clusters = pd.read_csv(cluster_csv)

required_candidate_columns = {
    'candidate_id', 'event_parity', 'channel', 'same_side',
    'side1', 'side2', 'charge1', 'charge2',
    'px1', 'py1', 'pz1', 'pt1', 'phi1', 'eta1',
    'px2', 'py2', 'pz2', 'pt2', 'phi2', 'eta2',
    'pca_x', 'pca_y', 'pca_z', 'selected_mass', 'pair_pt',
    'pass_cut03', 'pass_pion_pid', 'pass_signed_delta_phi',
}

required_cluster_columns = {
    'candidate_id', 'channel', 'daughter', 'track_id', 'charge', 'side',
    'layer', 'cluster_x', 'cluster_y', 'cluster_z',
    'cluster_r', 'cluster_phi',
    'fit_x', 'fit_y', 'fit_z', 'fit_px', 'fit_py', 'fit_pz',
    'residual_r', 'residual_rphi', 'residual_z',
    'assigned_sigma_r', 'assigned_sigma_rphi', 'assigned_sigma_z',
    'recommended_for_reference_fit', 'kalman_measurement_used',
}

missing_candidates = sorted(required_candidate_columns - set(candidates.columns))
missing_clusters = sorted(required_cluster_columns - set(clusters.columns))

if missing_candidates:
    raise KeyError(f'Missing candidate columns: {missing_candidates}')

if missing_clusters:
    raise KeyError(f'Missing cluster columns: {missing_clusters}')

# Fill uncertainty columns before constructing sigma_normal.
sigma_r = clusters['assigned_sigma_r'].to_numpy(float)
sigma_rphi = clusters['assigned_sigma_rphi'].to_numpy(float)
sigma_z = clusters['assigned_sigma_z'].to_numpy(float)

sigma_r[~np.isfinite(sigma_r) | (sigma_r <= 0)] = fallback_sigma_r_cm
sigma_rphi[~np.isfinite(sigma_rphi) | (sigma_rphi <= 0)] = fallback_sigma_rphi_cm
sigma_z[~np.isfinite(sigma_z) | (sigma_z <= 0)] = fallback_sigma_z_cm

clusters['sigma_r_filled'] = sigma_r
clusters['sigma_rphi_filled'] = sigma_rphi
clusters['sigma_z_filled'] = sigma_z

fit_px = clusters['fit_px'].to_numpy(float)
fit_py = clusters['fit_py'].to_numpy(float)
fit_pt = np.hypot(fit_px, fit_py)
valid_fit_tangent = np.isfinite(fit_pt) & (fit_pt > 1.0e-8)

tangent_x = np.full(len(clusters), np.nan, dtype=float)
tangent_y = np.full(len(clusters), np.nan, dtype=float)
tangent_x[valid_fit_tangent] = fit_px[valid_fit_tangent] / fit_pt[valid_fit_tangent]
tangent_y[valid_fit_tangent] = fit_py[valid_fit_tangent] / fit_pt[valid_fit_tangent]

normal_x = -tangent_y
normal_y = tangent_x

cluster_phi = clusters['cluster_phi'].to_numpy(float)
radial_x = np.cos(cluster_phi)
radial_y = np.sin(cluster_phi)
azimuthal_x = -np.sin(cluster_phi)
azimuthal_y = np.cos(cluster_phi)

normal_projection_r = normal_x * radial_x + normal_y * radial_y
normal_projection_rphi = normal_x * azimuthal_x + normal_y * azimuthal_y

displacement_x = (
    clusters['cluster_x'].to_numpy(float)
    - clusters['fit_x'].to_numpy(float)
)
displacement_y = (
    clusters['cluster_y'].to_numpy(float)
    - clusters['fit_y'].to_numpy(float)
)

residual_normal_xy = displacement_x * normal_x + displacement_y * normal_y

# These two projections are useful QA quantities, but the training target is
# the single normal observation, not two falsely independent components.
residual_r_orthogonal = residual_normal_xy * normal_projection_r
residual_rphi_orthogonal = residual_normal_xy * normal_projection_rphi

sigma_normal = np.sqrt(
    (normal_projection_r * sigma_r) ** 2
    + (normal_projection_rphi * sigma_rphi) ** 2
)
sigma_normal = np.maximum(sigma_normal, minimum_sigma_normal_cm)

clusters['normal_projection_r'] = normal_projection_r
clusters['normal_projection_rphi'] = normal_projection_rphi
clusters['residual_normal_xy'] = residual_normal_xy
clusters['residual_r_orthogonal'] = residual_r_orthogonal
clusters['residual_rphi_orthogonal'] = residual_rphi_orthogonal
clusters['sigma_normal'] = sigma_normal
clusters['drift_fraction'] = np.clip(
    1.0 - np.abs(clusters['cluster_z'].to_numpy(float)) / z_pad_cm,
    0.0,
    1.0,
)
clusters['distance_from_pad_plane_cm'] = np.maximum(
    0.0,
    z_pad_cm - np.abs(clusters['cluster_z'].to_numpy(float)),
)

side_values = clusters['side'].to_numpy(int)
z_values = clusters['cluster_z'].to_numpy(float)
physical_side_z_valid = (
    (
        (side_values == 0)
        & (z_values <= central_membrane_tolerance_cm)
        & (z_values >= -z_pad_cm - pad_plane_tolerance_cm)
    )
    | (
        (side_values == 1)
        & (z_values >= -central_membrane_tolerance_cm)
        & (z_values <= z_pad_cm + pad_plane_tolerance_cm)
    )
)
clusters['physical_side_z_valid'] = physical_side_z_valid.astype(np.uint8)

print('Candidate rows:', len(candidates))
print(candidates.groupby(['channel', 'side1', 'event_parity']).size())
print('Cluster rows:', len(clusters))
print(clusters.groupby(['channel', 'side', 'event_parity']).size())
print('Valid fitted tangents:', int(valid_fit_tangent.sum()), '/', len(clusters))
print('Physical side-z rows:', int(physical_side_z_valid.sum()), '/', len(clusters))


In [ ]:
# ============================================================
# Residual QA before training
# ============================================================

kshort_cluster_qa = clusters.loc[
    (clusters['channel'] == 0)
    & clusters['physical_side_z_valid'].astype(bool)
].copy()

figure, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)

qa_definitions = [
    ('residual_r', r'original residual_r [cm]', (-0.40, 0.40)),
    ('residual_r_orthogonal', r'orthogonal inferred $\Delta r$ [cm]', (-0.40, 0.40)),
    ('residual_normal_xy', r'transverse normal residual $d_n$ [cm]', (-0.60, 0.60)),
    ('residual_rphi', r'original $r\Delta\phi$ residual [cm]', (-0.60, 0.60)),
    ('residual_rphi_orthogonal', r'orthogonal inferred $r\Delta\phi$ [cm]', (-0.60, 0.60)),
    ('residual_z', r'$\Delta z$ residual [cm]', (-1.50, 1.50)),
]

for axis, (column, label, value_range) in zip(axes.flat, qa_definitions):
    for side in [0, 1]:
        values = kshort_cluster_qa.loc[
            kshort_cluster_qa['side'] == side,
            column,
        ].to_numpy(float)
        values = values[np.isfinite(values)]
        axis.hist(
            values,
            bins=160,
            range=value_range,
            histtype='step',
            linewidth=2,
            label=f'side {side}',
        )
    axis.set_xlabel(label)
    axis.set_ylabel('Clusters')
    axis.legend()

plt.show()

# Directly demonstrate why the original residual_r is not used.
print(
    kshort_cluster_qa.groupby('side')[[
        'residual_r',
        'residual_r_orthogonal',
        'residual_rphi',
        'residual_rphi_orthogonal',
        'residual_normal_xy',
        'residual_z',
    ]].std()
)


## Build padded candidate tensors and cache them once

Residual pretraining later flattens the valid clusters into large batches. Hybrid mass fine-tuning uses the padded candidate representation so both daughter tracks can be corrected and refitted together.


In [ ]:
# ============================================================
# Candidate-to-cluster tensor construction
# ============================================================

def clean_track_clusters(frame):
    frame = frame.copy()

    finite = (
        np.isfinite(frame['cluster_r'])
        & np.isfinite(frame['cluster_phi'])
        & np.isfinite(frame['cluster_z'])
        & np.isfinite(frame['residual_normal_xy'])
        & np.isfinite(frame['normal_projection_r'])
        & np.isfinite(frame['normal_projection_rphi'])
        & np.isfinite(frame['sigma_normal'])
        & np.isfinite(frame['residual_z'])
        & np.isfinite(frame['sigma_z_filled'])
    )

    frame = frame.loc[finite]
    frame = frame.loc[frame['physical_side_z_valid'].astype(bool)]
    frame = frame.loc[(frame['layer'] >= 7) & (frame['layer'] <= 54)]
    frame = frame.loc[
        (np.abs(frame['residual_normal_xy']) < maximum_abs_residual_normal_cm)
        & (np.abs(frame['residual_z']) < maximum_abs_residual_z_cm)
    ]

    if use_recommended_clusters_only:
        frame = frame.loc[
            frame['recommended_for_reference_fit'].astype(bool)
        ]

    if use_kalman_used_clusters_only:
        frame = frame.loc[
            frame['kalman_measurement_used'].astype(bool)
        ]

    sort_columns = ['layer']
    if 'kalman_measurement_chi2' in frame.columns:
        sort_columns.append('kalman_measurement_chi2')

    frame = frame.sort_values(sort_columns)
    frame = frame.drop_duplicates('layer', keep='first')

    return frame.sort_values('layer').head(maximum_clusters)


def smooth_pair_weights(pt1, pt2):
    if not use_smooth_track_pt_reweighting:
        return np.ones(len(pt1), dtype=np.float32)

    all_pt = np.concatenate([pt1, pt2]).astype(float)
    safe = np.clip(all_pt, 0.20, 5.00)
    edges = np.linspace(np.log(0.20), np.log(5.00), 81)
    centers = 0.5 * (edges[:-1] + edges[1:])
    counts, _ = np.histogram(np.log(safe), bins=edges)

    offsets = np.arange(-8, 9, dtype=float)
    kernel = np.exp(-0.5 * (offsets / 2.0) ** 2)
    kernel /= kernel.sum()

    density = np.convolve(counts.astype(float), kernel, mode='same')
    positive = density[density > 0]
    reference = np.median(positive) if len(positive) else 1.0
    density = np.maximum(density, 0.05 * max(reference, 1.0))

    weights = (reference / density) ** track_pt_weight_power
    weights = np.clip(weights, 0.35, 3.0)

    def evaluate(values):
        return np.interp(
            np.log(np.clip(values, 0.20, 5.00)),
            centers,
            weights,
            left=weights[0],
            right=weights[-1],
        )

    pair = np.sqrt(evaluate(pt1) * evaluate(pt2))
    pair /= np.mean(pair)

    return np.clip(
        pair,
        minimum_pair_weight,
        maximum_pair_weight,
    ).astype(np.float32)


def build_channel_arrays(channel, side):
    candidate_frame = candidates.loc[
        (candidates['channel'] == channel)
        & candidates['same_side'].astype(bool)
        & (candidates['side1'] == side)
        & (candidates['side2'] == side)
    ].copy()

    if channel == 0:
        candidate_frame = candidate_frame.loc[
            candidate_frame['pass_cut03'].astype(bool)
            & candidate_frame['pass_pion_pid'].astype(bool)
            & candidate_frame['pass_signed_delta_phi'].astype(bool)
        ]

    channel_clusters = clusters.loc[
        (clusters['channel'] == channel)
        & (clusters['side'] == side)
    ]

    grouped = {
        key: value
        for key, value in channel_clusters.groupby(
            ['candidate_id', 'daughter'],
            sort=False,
        )
    }

    records = []

    for row in candidate_frame.itertuples(index=False):
        key1 = (row.candidate_id, 1)
        key2 = (row.candidate_id, 2)

        if key1 not in grouped or key2 not in grouped:
            continue

        first = clean_track_clusters(grouped[key1])
        second = clean_track_clusters(grouped[key2])

        if (
            len(first) < minimum_clusters_per_track
            or len(second) < minimum_clusters_per_track
        ):
            continue

        records.append((row, first, second))

    if not records:
        raise RuntimeError(
            f'No usable candidates for channel={channel}, side={side}'
        )

    number = len(records)
    shape = (number, maximum_clusters)

    output = {
        key: np.zeros(shape, dtype=np.float32)
        for key in [
            'r1', 'phi1_cluster', 'z1_cluster',
            'res_normal1', 'normal_projection_r1', 'normal_projection_rphi1',
            'sigma_normal1', 'res_z1', 'sigma_z1', 'mask1',
            'r2', 'phi2_cluster', 'z2_cluster',
            'res_normal2', 'normal_projection_r2', 'normal_projection_rphi2',
            'sigma_normal2', 'res_z2', 'sigma_z2', 'mask2',
        ]
    }

    scalar_names = [
        'candidate_id', 'event_parity', 'selected_mass', 'pair_pt',
        'pca_x', 'pca_y', 'pca_z',
        'charge1', 'charge2', 'pt1', 'pt2', 'phi1', 'phi2', 'eta1', 'eta2',
        'px1', 'py1', 'pz1', 'px2', 'py2', 'pz2',
    ]

    for name in scalar_names:
        output[name] = np.zeros(number, dtype=np.float32)

    def fill_track(index, frame, suffix):
        count = len(frame)

        output[f'r{suffix}'][index, :count] = frame[
            'cluster_r'
        ].to_numpy(np.float32)

        output[f'phi{suffix}_cluster'][index, :count] = frame[
            'cluster_phi'
        ].to_numpy(np.float32)

        output[f'z{suffix}_cluster'][index, :count] = frame[
            'cluster_z'
        ].to_numpy(np.float32)

        output[f'res_normal{suffix}'][index, :count] = frame[
            'residual_normal_xy'
        ].to_numpy(np.float32)

        output[f'normal_projection_r{suffix}'][index, :count] = frame[
            'normal_projection_r'
        ].to_numpy(np.float32)

        output[f'normal_projection_rphi{suffix}'][index, :count] = frame[
            'normal_projection_rphi'
        ].to_numpy(np.float32)

        output[f'sigma_normal{suffix}'][index, :count] = frame[
            'sigma_normal'
        ].to_numpy(np.float32)

        output[f'res_z{suffix}'][index, :count] = frame[
            'residual_z'
        ].to_numpy(np.float32)

        output[f'sigma_z{suffix}'][index, :count] = frame[
            'sigma_z_filled'
        ].to_numpy(np.float32)

        output[f'mask{suffix}'][index, :count] = 1.0

    for index, (row, first, second) in enumerate(records):
        for name in scalar_names:
            output[name][index] = getattr(row, name)

        fill_track(index, first, '1')
        fill_track(index, second, '2')

    output['pair_weight'] = smooth_pair_weights(
        output['pt1'],
        output['pt2'],
    )

    output['channel'] = np.full(number, channel, dtype=np.int64)
    output['side'] = np.full(number, side, dtype=np.int64)

    print(
        f'channel={channel}, side={side}: '
        f'{number} usable candidates'
    )

    return output


kshort_arrays = {
    side: build_channel_arrays(0, side)
    for side in [0, 1]
}

validation_arrays = {}

for channel in [1, 2]:
    for side in [0, 1]:
        try:
            validation_arrays[(channel, side)] = build_channel_arrays(
                channel,
                side,
            )
        except RuntimeError as error:
            print(error)


In [ ]:
# ============================================================
# Cache side arrays for parallel workers
# ============================================================

kshort_npz_files = {}
validation_npz_files = {}

for side, arrays in kshort_arrays.items():
    output_file = cache_dir / f'kshort_side{side}.npz'
    np.savez_compressed(output_file, **arrays)
    kshort_npz_files[side] = output_file
    print('Wrote:', output_file)

for (channel, side), arrays in validation_arrays.items():
    output_file = cache_dir / f'channel{channel}_side{side}.npz'
    np.savez_compressed(output_file, **arrays)
    validation_npz_files[(channel, side)] = output_file
    print('Wrote:', output_file)


def load_npz_dictionary(path):
    with np.load(path, allow_pickle=False) as input_file:
        return {
            key: input_file[key]
            for key in input_file.files
        }


In [ ]:
# ============================================================
# PyTorch datasets
# ============================================================

class CandidateDataset(Dataset):
    def __init__(self, arrays, indices):
        self.data = {}
        number_of_candidates = len(arrays['candidate_id'])

        for key, values in arrays.items():
            if (
                isinstance(values, np.ndarray)
                and len(values) == number_of_candidates
            ):
                self.data[key] = torch.from_numpy(values[indices])

    def __len__(self):
        return len(self.data['candidate_id'])

    def __getitem__(self, index):
        return {
            key: value[index]
            for key, value in self.data.items()
        }


class ResidualDataset(Dataset):
    def __init__(self, arrays):
        self.data = {
            key: torch.from_numpy(value)
            for key, value in arrays.items()
        }

    def __len__(self):
        return len(self.data['r'])

    def __getitem__(self, index):
        return {
            key: value[index]
            for key, value in self.data.items()
        }


def flatten_training_clusters(arrays, candidate_indices):
    output_lists = {
        key: []
        for key in [
            'r', 'phi', 'z',
            'res_normal',
            'normal_projection_r',
            'normal_projection_rphi',
            'sigma_normal',
            'res_z', 'sigma_z',
        ]
    }

    for suffix in ['1', '2']:
        mask = arrays[f'mask{suffix}'][candidate_indices] > 0.5

        source_names = {
            'r': f'r{suffix}',
            'phi': f'phi{suffix}_cluster',
            'z': f'z{suffix}_cluster',
            'res_normal': f'res_normal{suffix}',
            'normal_projection_r': f'normal_projection_r{suffix}',
            'normal_projection_rphi': f'normal_projection_rphi{suffix}',
            'sigma_normal': f'sigma_normal{suffix}',
            'res_z': f'res_z{suffix}',
            'sigma_z': f'sigma_z{suffix}',
        }

        for output_name, source_name in source_names.items():
            output_lists[output_name].append(
                arrays[source_name][candidate_indices][mask].astype(np.float32)
            )

    output = {
        key: np.concatenate(value).astype(np.float32)
        for key, value in output_lists.items()
    }

    return output


## Smooth side-specific detector field with a hard pad-plane boundary

Each side has an independent model and therefore can have a different value at \(z=0\). The model receives smooth radial basis functions, periodic azimuthal harmonics, and the drift fraction \(u\). Its output is multiplied by \(u\), so all components are exactly zero at the corresponding pad plane.


In [ ]:
# ============================================================
# Spatial network and differentiable refit helpers
# ============================================================

class SpatialResidualNetwork(nn.Module):
    def __init__(self, side):
        super().__init__()

        if side not in (0, 1):
            raise ValueError('side must be 0 or 1')

        self.side = int(side)

        radial_centers = torch.linspace(
            radial_basis_minimum_cm,
            radial_basis_maximum_cm,
            number_of_radial_basis_functions,
        )

        radial_spacing = (
            radial_basis_maximum_cm
            - radial_basis_minimum_cm
        ) / max(number_of_radial_basis_functions - 1, 1)

        self.register_buffer('radial_centers', radial_centers)
        self.radial_width = radial_basis_width_scale * radial_spacing

        input_width = (
            number_of_radial_basis_functions
            + 2 * number_of_phi_harmonics
            + 2
        )

        layers = []
        width = input_width

        for _ in range(hidden_layers):
            layers.extend([
                nn.Linear(width, hidden_width),
                nn.SiLU(),
                nn.Dropout(dropout_probability),
            ])
            width = hidden_width

        self.trunk = nn.Sequential(*layers)
        self.output = nn.Linear(
            width,
            3 if include_z_correction else 2,
        )

        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def physical_domain_and_drift_fraction(self, z):
        if self.side == 0:
            physical = (
                (z <= central_membrane_tolerance_cm)
                & (z >= -z_pad_cm - pad_plane_tolerance_cm)
            )
        else:
            physical = (
                (z >= -central_membrane_tolerance_cm)
                & (z <= z_pad_cm + pad_plane_tolerance_cm)
            )

        drift_fraction = torch.clamp(
            1.0 - torch.abs(z) / z_pad_cm,
            min=0.0,
            max=1.0,
        )

        return physical.to(z.dtype), drift_fraction

    def features(self, r, phi, z):
        radial = torch.exp(
            -0.5
            * (
                (
                    r[..., None]
                    - self.radial_centers
                )
                / self.radial_width
            ) ** 2
        )

        phi_features = []

        for harmonic in range(1, number_of_phi_harmonics + 1):
            phi_features.extend([
                torch.sin(harmonic * phi)[..., None],
                torch.cos(harmonic * phi)[..., None],
            ])

        _, drift_fraction = self.physical_domain_and_drift_fraction(z)

        return torch.cat([
            radial,
            *phi_features,
            drift_fraction[..., None],
            (drift_fraction ** 2)[..., None],
        ], dim=-1)

    def forward(self, r, phi, z):
        physical, drift_fraction = self.physical_domain_and_drift_fraction(z)
        feature_tensor = self.features(r, phi, z)
        original_shape = r.shape

        raw = self.output(
            self.trunk(
                feature_tensor.reshape(
                    -1,
                    feature_tensor.shape[-1],
                )
            )
        ).reshape(*original_shape, -1)

        boundary = drift_fraction * physical

        delta_r = (
            boundary
            * maximum_delta_r_cm
            * torch.tanh(raw[..., 0])
        )

        r_delta_phi = (
            boundary
            * maximum_r_delta_phi_cm
            * torch.tanh(raw[..., 1])
        )

        if include_z_correction:
            delta_z = (
                boundary
                * maximum_delta_z_cm
                * torch.tanh(raw[..., 2])
            )
        else:
            delta_z = torch.zeros_like(delta_r)

        return delta_r, r_delta_phi, delta_z


def wrap_phi_torch(value):
    return torch.atan2(
        torch.sin(value),
        torch.cos(value),
    )


def masked_circle_fit(x, y, mask):
    ones = torch.ones_like(x)
    design = torch.stack([x, y, ones], dim=2)
    target = -(x * x + y * y)[:, :, None]
    weight = mask[:, :, None]

    weighted_design = design * weight
    normal = design.transpose(1, 2) @ weighted_design
    rhs = design.transpose(1, 2) @ (target * weight)

    identity = torch.eye(
        3,
        dtype=x.dtype,
        device=x.device,
    )[None]

    parameters = torch.linalg.solve(
        normal + 1.0e-7 * identity,
        rhs,
    )[:, :, 0]

    coefficient_a = parameters[:, 0]
    coefficient_b = parameters[:, 1]
    coefficient_c = parameters[:, 2]

    center_x = -0.5 * coefficient_a
    center_y = -0.5 * coefficient_b

    radius = torch.sqrt(
        torch.clamp(
            0.25 * (
                coefficient_a ** 2
                + coefficient_b ** 2
            ) - coefficient_c,
            min=1.0e-8,
        )
    )

    return center_x, center_y, radius


def tangent_phi_at_vertex(
    center_x,
    center_y,
    radius,
    charge,
    vertex_x,
    vertex_y,
):
    dx = vertex_x - center_x
    dy = vertex_y - center_y

    distance = torch.sqrt(
        torch.clamp(
            dx * dx + dy * dy,
            min=1.0e-12,
        )
    )

    point_x = center_x + radius * dx / distance
    point_y = center_y + radius * dy / distance

    omega = -charge / radius
    sin_phi = omega * (point_x - center_x)
    cos_phi = -omega * (point_y - center_y)

    return torch.atan2(sin_phi, cos_phi)


def longitudinal_slope(x, y, z, mask, center_x, center_y, radius):
    theta = torch.atan2(
        y - center_y[:, None],
        x - center_x[:, None],
    )

    delta = wrap_phi_torch(
        theta[:, 1:] - theta[:, :-1]
    )

    pair_mask = mask[:, 1:] * mask[:, :-1]
    ds = radius[:, None] * torch.abs(delta) * pair_mask

    path_length = torch.cat([
        torch.zeros_like(ds[:, :1]),
        torch.cumsum(ds, dim=1),
    ], dim=1)

    weight_sum = torch.clamp(mask.sum(dim=1), min=1.0)
    mean_s = (path_length * mask).sum(dim=1) / weight_sum
    mean_z = (z * mask).sum(dim=1) / weight_sum

    covariance = (
        (path_length - mean_s[:, None])
        * (z - mean_z[:, None])
        * mask
    ).sum(dim=1)

    variance = (
        (path_length - mean_s[:, None]) ** 2
        * mask
    ).sum(dim=1)

    return covariance / torch.clamp(variance, min=1.0e-8)


def refit_track(
    model,
    r,
    phi,
    z,
    mask,
    charge,
    original_pt,
    original_phi,
    original_eta,
    vertex_x,
    vertex_y,
):
    raw_x = r * torch.cos(phi)
    raw_y = r * torch.sin(phi)

    raw_center_x, raw_center_y, raw_radius = masked_circle_fit(
        raw_x,
        raw_y,
        mask,
    )

    raw_fit_phi = tangent_phi_at_vertex(
        raw_center_x,
        raw_center_y,
        raw_radius,
        charge,
        vertex_x,
        vertex_y,
    )

    raw_slope = longitudinal_slope(
        raw_x,
        raw_y,
        z,
        mask,
        raw_center_x,
        raw_center_y,
        raw_radius,
    )

    delta_r, r_delta_phi, delta_z = model(r, phi, z)

    corrected_r = r - delta_r * mask
    corrected_phi = (
        phi
        - (
            r_delta_phi
            / torch.clamp(r, min=1.0e-6)
        ) * mask
    )
    corrected_z = z - delta_z * mask

    corrected_x = corrected_r * torch.cos(corrected_phi)
    corrected_y = corrected_r * torch.sin(corrected_phi)

    center_x, center_y, radius = masked_circle_fit(
        corrected_x,
        corrected_y,
        mask,
    )

    fitted_phi = tangent_phi_at_vertex(
        center_x,
        center_y,
        radius,
        charge,
        vertex_x,
        vertex_y,
    )

    fitted_slope = longitudinal_slope(
        corrected_x,
        corrected_y,
        corrected_z,
        mask,
        center_x,
        center_y,
        radius,
    )

    corrected_pt = (
        original_pt
        * radius
        / torch.clamp(raw_radius, min=1.0e-6)
    )

    corrected_momentum_phi = (
        original_phi
        + wrap_phi_torch(
            fitted_phi - raw_fit_phi
        )
    )

    corrected_sinh_eta = (
        torch.sinh(original_eta)
        + (fitted_slope - raw_slope)
    )

    px = corrected_pt * torch.cos(corrected_momentum_phi)
    py = corrected_pt * torch.sin(corrected_momentum_phi)
    pz = corrected_pt * corrected_sinh_eta

    momentum = torch.stack([px, py, pz], dim=1)

    return {
        'momentum': momentum,
        'pt': corrected_pt,
        'phi': corrected_momentum_phi,
        'delta_r': delta_r,
        'r_delta_phi': r_delta_phi,
        'delta_z': delta_z,
    }


def invariant_mass(momentum1, momentum2, mass1, mass2):
    energy1 = torch.sqrt(
        torch.sum(momentum1 ** 2, dim=1)
        + mass1 ** 2
    )

    energy2 = torch.sqrt(
        torch.sum(momentum2 ** 2, dim=1)
        + mass2 ** 2
    )

    total = momentum1 + momentum2

    mass_squared = (
        (energy1 + energy2) ** 2
        - torch.sum(total ** 2, dim=1)
    )

    return torch.sqrt(
        torch.clamp(mass_squared, min=0.0)
    )


In [ ]:
# ============================================================
# Mass prefit and likelihood
# ============================================================

SQRT_TWO = math.sqrt(2.0)


def cdf_scalar(value):
    return 0.5 * (
        1.0
        + math.erf(float(value) / SQRT_TWO)
    )


def mass_pdf_numpy(
    mass,
    fraction,
    mean,
    sigma,
    slope,
    minimum,
    maximum,
):
    mass = np.asarray(mass, dtype=float)

    lower = cdf_scalar((minimum - mean) / sigma)
    upper = cdf_scalar((maximum - mean) / sigma)

    normalization = (
        sigma
        * math.sqrt(2.0 * math.pi)
        * max(upper - lower, 1.0e-12)
    )

    signal = np.exp(
        -0.5 * ((mass - mean) / sigma) ** 2
    ) / normalization

    normalized_mass = (
        2.0 * mass - maximum - minimum
    ) / (maximum - minimum)

    background = np.clip(
        (1.0 + slope * normalized_mass)
        / (maximum - minimum),
        1.0e-12,
        None,
    )

    return (
        fraction * signal
        + (1.0 - fraction) * background
    )


def fit_mass_model(values, minimum=0.44, maximum=0.56):
    values = np.asarray(values, dtype=float)
    values = values[
        (values >= minimum)
        & (values <= maximum)
    ]

    def objective(parameters):
        fraction, mean, sigma, slope = parameters
        probability = mass_pdf_numpy(
            values,
            fraction,
            mean,
            sigma,
            slope,
            minimum,
            maximum,
        )

        return -np.sum(
            np.log(
                np.clip(probability, 1.0e-15, None)
            )
        )

    result = minimize(
        objective,
        [0.55, 0.495, 0.009, 0.0],
        method='L-BFGS-B',
        bounds=[
            (0.05, 0.95),
            (0.475, 0.515),
            (0.003, 0.025),
            (-0.9, 0.9),
        ],
    )

    return {
        'signal_fraction': float(result.x[0]),
        'raw_mean': float(result.x[1]),
        'signal_sigma': float(result.x[2]),
        'background_slope': float(result.x[3]),
        'mass_minimum': minimum,
        'mass_maximum': maximum,
    }


mass_models = {}

for side in [0, 1]:
    arrays = kshort_arrays[side]
    training_mask = (
        arrays['event_parity'].astype(int)
        == training_parity
    )

    mass_models[side] = fit_mass_model(
        arrays['selected_mass'][training_mask]
    )

    print('side', side, mass_models[side])


def mass_nll_per_candidate(mass, model):
    fraction = model['signal_fraction']
    sigma = model['signal_sigma']
    slope = model['background_slope']
    minimum = model['mass_minimum']
    maximum = model['mass_maximum']

    lower = cdf_scalar(
        (minimum - pdg_kshort_mass) / sigma
    )
    upper = cdf_scalar(
        (maximum - pdg_kshort_mass) / sigma
    )

    normalization = (
        sigma
        * math.sqrt(2.0 * math.pi)
        * max(upper - lower, 1.0e-12)
    )

    signal = torch.exp(
        -0.5
        * ((mass - pdg_kshort_mass) / sigma) ** 2
    ) / normalization

    normalized_mass = (
        2.0 * mass - maximum - minimum
    ) / (maximum - minimum)

    background = torch.clamp(
        (1.0 + slope * normalized_mass)
        / (maximum - minimum),
        min=1.0e-12,
    )

    probability = (
        fraction * signal
        + (1.0 - fraction) * background
    )

    return -torch.log(
        torch.clamp(probability, min=1.0e-12)
    )


In [ ]:
# ============================================================
# Batch correction and losses
# ============================================================

def correct_batch(model, batch, channel=0):
    first = refit_track(
        model,
        batch['r1'],
        batch['phi1_cluster'],
        batch['z1_cluster'],
        batch['mask1'],
        batch['charge1'],
        batch['pt1'],
        batch['phi1'],
        batch['eta1'],
        batch['pca_x'],
        batch['pca_y'],
    )

    second = refit_track(
        model,
        batch['r2'],
        batch['phi2_cluster'],
        batch['z2_cluster'],
        batch['mask2'],
        batch['charge2'],
        batch['pt2'],
        batch['phi2'],
        batch['eta2'],
        batch['pca_x'],
        batch['pca_y'],
    )

    if channel == 0:
        mass1 = mass2 = pion_mass
    elif channel == 1:
        mass1 = torch.where(
            batch['charge1'] > 0,
            torch.full_like(batch['charge1'], proton_mass),
            torch.full_like(batch['charge1'], pion_mass),
        )
        mass2 = torch.where(
            batch['charge2'] > 0,
            torch.full_like(batch['charge2'], proton_mass),
            torch.full_like(batch['charge2'], pion_mass),
        )
    else:
        mass1 = torch.where(
            batch['charge1'] < 0,
            torch.full_like(batch['charge1'], proton_mass),
            torch.full_like(batch['charge1'], pion_mass),
        )
        mass2 = torch.where(
            batch['charge2'] < 0,
            torch.full_like(batch['charge2'], proton_mass),
            torch.full_like(batch['charge2'], pion_mass),
        )

    mass = invariant_mass(
        first['momentum'],
        second['momentum'],
        mass1,
        mass2,
    )

    return first, second, mass


def masked_huber(prediction, target, sigma, mask=None):
    normalized = (
        prediction - target
    ) / torch.clamp(sigma, min=1.0e-4)

    loss = F.smooth_l1_loss(
        normalized,
        torch.zeros_like(normalized),
        reduction='none',
    )

    if mask is None:
        return torch.mean(loss)

    return torch.sum(loss * mask) / torch.clamp(
        torch.sum(mask),
        min=1.0,
    )


def residual_batch_loss(model, batch):
    delta_r, r_delta_phi, delta_z = model(
        batch['r'],
        batch['phi'],
        batch['z'],
    )

    predicted_normal = (
        delta_r * batch['normal_projection_r']
        + r_delta_phi * batch['normal_projection_rphi']
    )

    loss = masked_huber(
        predicted_normal,
        batch['res_normal'],
        batch['sigma_normal'],
    )

    if include_z_correction:
        loss = loss + masked_huber(
            delta_z,
            batch['res_z'],
            batch['sigma_z'],
        )

    return loss


def candidate_residual_loss(model, batch):
    losses = []

    for suffix in ['1', '2']:
        delta_r, r_delta_phi, delta_z = model(
            batch[f'r{suffix}'],
            batch[f'phi{suffix}_cluster'],
            batch[f'z{suffix}_cluster'],
        )

        predicted_normal = (
            delta_r * batch[f'normal_projection_r{suffix}']
            + r_delta_phi * batch[f'normal_projection_rphi{suffix}']
        )

        track_loss = masked_huber(
            predicted_normal,
            batch[f'res_normal{suffix}'],
            batch[f'sigma_normal{suffix}'],
            batch[f'mask{suffix}'],
        )

        if include_z_correction:
            track_loss = track_loss + masked_huber(
                delta_z,
                batch[f'res_z{suffix}'],
                batch[f'sigma_z{suffix}'],
                batch[f'mask{suffix}'],
            )

        losses.append(track_loss)

    return 0.5 * (losses[0] + losses[1])


def regularization_losses(model, batch):
    r = torch.cat([batch['r1'], batch['r2']], dim=0)
    phi = torch.cat([
        batch['phi1_cluster'],
        batch['phi2_cluster'],
    ], dim=0)
    z = torch.cat([
        batch['z1_cluster'],
        batch['z2_cluster'],
    ], dim=0)
    mask = torch.cat([batch['mask1'], batch['mask2']], dim=0)

    delta_r, r_delta_phi, delta_z = model(r, phi, z)

    normalization = torch.clamp(mask.sum(), min=1.0)

    amplitude = torch.sum(
        (
            (delta_r / 0.15) ** 2
            + (r_delta_phi / 0.10) ** 2
        ) * mask
    ) / normalization

    if include_z_correction:
        amplitude = amplitude + torch.sum(
            (delta_z / 0.20) ** 2 * mask
        ) / normalization

    perturbed_r = torch.clamp(
        r + 0.25 * torch.randn_like(r),
        radial_basis_minimum_cm,
        radial_basis_maximum_cm,
    )

    perturbed_phi = phi + 0.005 * torch.randn_like(phi)

    perturbed_z = z + 0.50 * torch.randn_like(z)
    if model.side == 0:
        perturbed_z = torch.clamp(perturbed_z, -z_pad_cm, 0.0)
    else:
        perturbed_z = torch.clamp(perturbed_z, 0.0, z_pad_cm)

    delta_r_noise, r_delta_phi_noise, delta_z_noise = model(
        perturbed_r,
        perturbed_phi,
        perturbed_z,
    )

    smoothness = torch.sum(
        ((delta_r_noise - delta_r) / 0.025) ** 2 * mask
    ) / normalization

    smoothness = smoothness + torch.sum(
        ((r_delta_phi_noise - r_delta_phi) / 0.020) ** 2 * mask
    ) / normalization

    if include_z_correction:
        smoothness = smoothness + torch.sum(
            ((delta_z_noise - delta_z) / 0.050) ** 2 * mask
        ) / normalization

    mean_field = (
        torch.sum(delta_r * mask)
        / normalization
        / 0.15
    ) ** 2

    mean_field = mean_field + (
        torch.sum(r_delta_phi * mask)
        / normalization
        / 0.10
    ) ** 2

    if include_z_correction:
        mean_field = mean_field + (
            torch.sum(delta_z * mask)
            / normalization
            / 0.20
        ) ** 2

    return amplitude, smoothness, mean_field


def weighted_mean(values, weights):
    return torch.sum(values * weights) / torch.clamp(
        torch.sum(weights),
        min=1.0e-12,
    )


In [ ]:
# ============================================================
# Zero-map mass closure and hard pad-plane boundary tests
# ============================================================

for side in [0, 1]:
    arrays = kshort_arrays[side]
    indices = np.arange(min(64, len(arrays['candidate_id'])))

    batch = next(iter(DataLoader(
        CandidateDataset(arrays, indices),
        batch_size=len(indices),
    )))

    model = SpatialResidualNetwork(side)

    _, _, corrected_mass = correct_batch(
        model,
        batch,
        channel=0,
    )

    momentum1 = torch.stack([
        batch['px1'],
        batch['py1'],
        batch['pz1'],
    ], dim=1)

    momentum2 = torch.stack([
        batch['px2'],
        batch['py2'],
        batch['pz2'],
    ], dim=1)

    original_mass = invariant_mass(
        momentum1,
        momentum2,
        pion_mass,
        pion_mass,
    )

    print(
        'side', side,
        'maximum zero-map mass closure =',
        float(torch.max(torch.abs(corrected_mass - original_mass))),
    )

    test_r = torch.full((100,), 53.0)
    test_phi = torch.linspace(-math.pi, math.pi, 100)
    test_z = torch.full(
        (100,),
        -z_pad_cm if side == 0 else z_pad_cm,
    )

    delta_r, r_delta_phi, delta_z = model(
        test_r,
        test_phi,
        test_z,
    )

    print(
        'side', side,
        'pad-plane maxima:',
        float(torch.max(torch.abs(delta_r))),
        float(torch.max(torch.abs(r_delta_phi))),
        float(torch.max(torch.abs(delta_z))),
    )


In [ ]:
# ============================================================
# Validation metrics
# ============================================================

def peak_metrics(values, minimum=0.47, maximum=0.53):
    values = np.asarray(values, dtype=float)
    core = values[
        (values >= minimum)
        & (values <= maximum)
    ]

    return {
        'mean': float(np.mean(core)) if len(core) else np.nan,
        'width': (
            float(np.std(core, ddof=1))
            if len(core) > 1
            else np.nan
        ),
        'entries': int(len(core)),
    }


@torch.no_grad()
def evaluate_model(model, loader, side, channel=0):
    model.eval()
    before = []
    after = []

    for batch in loader:
        _, _, mass = correct_batch(
            model,
            batch,
            channel=channel,
        )

        before.append(batch['selected_mass'].numpy())
        after.append(mass.numpy())

    before = np.concatenate(before)
    after = np.concatenate(after)

    if channel == 0:
        before_metrics = peak_metrics(before, 0.47, 0.53)
        after_metrics = peak_metrics(after, 0.47, 0.53)
    else:
        before_metrics = peak_metrics(before, 1.09, 1.14)
        after_metrics = peak_metrics(after, 1.09, 1.14)

    if channel == 0:
        pdf = mass_pdf_numpy(
            after,
            mass_models[side]['signal_fraction'],
            pdg_kshort_mass,
            mass_models[side]['signal_sigma'],
            mass_models[side]['background_slope'],
            mass_models[side]['mass_minimum'],
            mass_models[side]['mass_maximum'],
        )

        nll = -float(
            np.mean(
                np.log(
                    np.clip(pdf, 1.0e-15, None)
                )
            )
        )

        mean_pull = 1000.0 * abs(
            after_metrics['mean'] - pdg_kshort_mass
        )

        width_ratio = (
            after_metrics['width']
            / before_metrics['width']
        )

        score = (
            nll
            + 0.25 * mean_pull
            + 2.0 * max(0.0, width_ratio - 1.0)
        )
    else:
        nll = np.nan
        width_ratio = (
            after_metrics['width']
            / before_metrics['width']
        )
        score = (
            abs(after_metrics['mean'] - before_metrics['mean'])
            + max(0.0, width_ratio - 1.0)
        )

    return {
        'score': float(score),
        'nll': nll,
        'before': before_metrics,
        'after': after_metrics,
        'width_ratio': float(width_ratio),
        'mass_before': before,
        'mass_after': after,
    }


## Parallel training

The flattened residual stage uses `residual_batch_size` clusters per optimizer step. The candidate mass stage uses `candidate_batch_size` complete V0 candidates.

For a 12-CPU allocation, the default launches:

```text
side 0 worker: 6 PyTorch/BLAS threads
side 1 worker: 6 PyTorch/BLAS threads
```

Each ensemble round trains the two sides simultaneously.


In [ ]:
# ============================================================
# Parallel side-training workers
# ============================================================

def configure_worker_threads(number_of_threads):
    os.environ['OMP_NUM_THREADS'] = str(number_of_threads)
    os.environ['MKL_NUM_THREADS'] = str(number_of_threads)
    os.environ['OPENBLAS_NUM_THREADS'] = str(number_of_threads)
    os.environ['NUMEXPR_NUM_THREADS'] = str(number_of_threads)

    torch.set_num_threads(number_of_threads)

    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass


def train_side_worker(side, ensemble_index):
    started = time.time()

    try:
        configure_worker_threads(threads_per_side)

        model_seed = (
            seed
            + 10000 * side
            + ensemble_index
        )

        random.seed(model_seed)
        np.random.seed(model_seed)
        torch.manual_seed(model_seed)

        arrays = load_npz_dictionary(
            kshort_npz_files[side]
        )

        training_indices = np.flatnonzero(
            arrays['event_parity'].astype(int)
            == training_parity
        )

        validation_indices = np.flatnonzero(
            arrays['event_parity'].astype(int)
            == validation_parity
        )

        if len(training_indices) == 0 or len(validation_indices) == 0:
            raise RuntimeError(
                f'Empty train/validation split for side {side}'
            )

        residual_arrays = flatten_training_clusters(
            arrays,
            training_indices,
        )

        residual_loader = DataLoader(
            ResidualDataset(residual_arrays),
            batch_size=min(
                residual_batch_size,
                len(residual_arrays['r']),
            ),
            shuffle=True,
            num_workers=0,
            drop_last=False,
        )

        training_loader = DataLoader(
            CandidateDataset(arrays, training_indices),
            batch_size=min(
                candidate_batch_size,
                len(training_indices),
            ),
            shuffle=True,
            num_workers=0,
            drop_last=False,
        )

        validation_loader = DataLoader(
            CandidateDataset(arrays, validation_indices),
            batch_size=min(
                validation_batch_size,
                len(validation_indices),
            ),
            shuffle=False,
            num_workers=0,
            drop_last=False,
        )

        model = SpatialResidualNetwork(side)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )

        # ----------------------------------------------------
        # Stage A: large-batch cluster residual pretraining
        # ----------------------------------------------------
        residual_history = []

        for epoch in range(residual_pretraining_epochs):
            model.train()
            epoch_loss = 0.0
            number_of_batches = 0

            for batch in residual_loader:
                optimizer.zero_grad(set_to_none=True)

                loss = residual_batch_loss(
                    model,
                    batch,
                )

                if not torch.isfinite(loss):
                    raise FloatingPointError(
                        'Non-finite residual pretraining loss'
                    )

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    5.0,
                )

                optimizer.step()

                epoch_loss += float(loss)
                number_of_batches += 1

            average_loss = epoch_loss / max(number_of_batches, 1)
            residual_history.append({
                'epoch': epoch,
                'residual_loss': average_loss,
            })

            if epoch % 20 == 0:
                print(
                    f'[side {side}, ensemble {ensemble_index}] '
                    f'residual epoch {epoch}: '
                    f'{average_loss:.6f}'
                )

        # ----------------------------------------------------
        # Stage B: candidate-level hybrid fine-tuning
        # ----------------------------------------------------
        best_state = copy.deepcopy(model.state_dict())
        best_score = np.inf
        best_epoch = -1
        stale_epochs = 0
        hybrid_history = []

        for epoch in range(hybrid_training_epochs):
            model.train()

            component_sums = np.zeros(6, dtype=float)
            number_of_batches = 0

            for batch in training_loader:
                optimizer.zero_grad(set_to_none=True)

                _, _, mass = correct_batch(
                    model,
                    batch,
                    channel=0,
                )

                mass_loss = weighted_mean(
                    mass_nll_per_candidate(
                        mass,
                        mass_models[side],
                    ),
                    batch['pair_weight'],
                )

                residual_loss_value = candidate_residual_loss(
                    model,
                    batch,
                )

                amplitude, smoothness, mean_field = regularization_losses(
                    model,
                    batch,
                )

                total_loss = (
                    mass_loss_weight * mass_loss
                    + residual_loss_weight * residual_loss_value
                    + amplitude_loss_weight * amplitude
                    + smoothness_loss_weight * smoothness
                    + mean_field_loss_weight * mean_field
                )

                if not torch.isfinite(total_loss):
                    raise FloatingPointError(
                        'Non-finite hybrid loss'
                    )

                total_loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    5.0,
                )

                optimizer.step()

                component_sums += [
                    float(total_loss),
                    float(mass_loss),
                    float(residual_loss_value),
                    float(smoothness),
                    float(amplitude),
                    float(mean_field),
                ]

                number_of_batches += 1

            if (
                epoch % validation_interval == 0
                or epoch == hybrid_training_epochs - 1
            ):
                metrics = evaluate_model(
                    model,
                    validation_loader,
                    side,
                    channel=0,
                )

                averages = component_sums / max(number_of_batches, 1)

                row = {
                    'epoch': epoch,
                    'score': metrics['score'],
                    'mean': metrics['after']['mean'],
                    'width': metrics['after']['width'],
                    'width_ratio': metrics['width_ratio'],
                    'loss': averages[0],
                    'mass_loss': averages[1],
                    'residual_loss': averages[2],
                    'smoothness_loss': averages[3],
                    'amplitude_loss': averages[4],
                    'mean_field_loss': averages[5],
                }

                hybrid_history.append(row)

                print(
                    f'[side {side}, ensemble {ensemble_index}] '
                    f'hybrid:',
                    row,
                )

                if metrics['score'] < best_score - 1.0e-5:
                    best_score = metrics['score']
                    best_epoch = epoch
                    best_state = copy.deepcopy(model.state_dict())
                    stale_epochs = 0
                else:
                    stale_epochs += validation_interval

                if stale_epochs >= patience:
                    break

        model.load_state_dict(best_state)

        final_metrics = evaluate_model(
            model,
            validation_loader,
            side,
            channel=0,
        )

        checkpoint_path = (
            checkpoint_dir
            / f'spatial_model_side{side}_ensemble{ensemble_index}.pt'
        )

        torch.save({
            'state_dict': model.state_dict(),
            'include_z_correction': include_z_correction,
            'side': side,
            'ensemble_index': ensemble_index,
            'model_seed': model_seed,
            'best_epoch': best_epoch,
            'best_score': best_score,
            'configuration': {
                'z_pad_cm': z_pad_cm,
                'number_of_radial_basis_functions': number_of_radial_basis_functions,
                'number_of_phi_harmonics': number_of_phi_harmonics,
                'hidden_width': hidden_width,
                'hidden_layers': hidden_layers,
                'maximum_delta_r_cm': maximum_delta_r_cm,
                'maximum_r_delta_phi_cm': maximum_r_delta_phi_cm,
                'maximum_delta_z_cm': maximum_delta_z_cm,
            },
        }, checkpoint_path)

        pd.DataFrame(residual_history).to_csv(
            work_dir
            / f'residual_history_side{side}_ensemble{ensemble_index}.csv',
            index=False,
        )

        pd.DataFrame(hybrid_history).to_csv(
            work_dir
            / f'hybrid_history_side{side}_ensemble{ensemble_index}.csv',
            index=False,
        )

        return {
            'success': True,
            'side': side,
            'ensemble_index': ensemble_index,
            'checkpoint': str(checkpoint_path),
            'elapsed_seconds': time.time() - started,
            'best_epoch': best_epoch,
            'score': final_metrics['score'],
            'mean_before': final_metrics['before']['mean'],
            'mean_after': final_metrics['after']['mean'],
            'width_before': final_metrics['before']['width'],
            'width_after': final_metrics['after']['width'],
            'width_ratio': final_metrics['width_ratio'],
            'training_candidates': len(training_indices),
            'validation_candidates': len(validation_indices),
            'training_clusters': len(residual_arrays['r']),
        }

    except Exception as error:
        return {
            'success': False,
            'side': side,
            'ensemble_index': ensemble_index,
            'checkpoint': '',
            'elapsed_seconds': time.time() - started,
            'score': np.nan,
            'error': repr(error),
            'traceback': traceback.format_exc(),
        }


training_tasks = [
    (side, ensemble_index)
    for ensemble_index in range(number_of_ensemble_models)
    for side in [0, 1]
]

print('Training task order:', training_tasks)

if joblib_available:
    with parallel_backend(
        'loky',
        inner_max_num_threads=threads_per_side,
    ):
        training_results = Parallel(
            n_jobs=parallel_side_jobs,
            verbose=10,
            batch_size=1,
        )(
            delayed(train_side_worker)(side, ensemble_index)
            for side, ensemble_index in training_tasks
        )
else:
    training_results = [
        train_side_worker(side, ensemble_index)
        for side, ensemble_index in training_tasks
    ]

training_results_frame = pd.DataFrame(training_results)
training_results_frame.to_csv(
    work_dir / 'parallel_training_results.csv',
    index=False,
)

display(training_results_frame)

failed = training_results_frame.loc[
    ~training_results_frame['success'].astype(bool)
]

if len(failed):
    print(failed[['side', 'ensemble_index', 'error']])
    print(failed.iloc[0]['traceback'])
    raise RuntimeError('One or more side-training workers failed.')


In [ ]:
# ============================================================
# Load side ensembles
# ============================================================

trained_models = {0: [], 1: []}

for result in training_results:
    checkpoint = torch.load(
        result['checkpoint'],
        map_location='cpu',
        weights_only=False,
    )

    side = int(checkpoint['side'])
    model = SpatialResidualNetwork(side)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    trained_models[side].append(model)


class EnsembleSpatialModel(nn.Module):
    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleList(models)
        self.side = models[0].side

    def forward(self, r, phi, z):
        outputs = [
            model(r, phi, z)
            for model in self.models
        ]

        delta_r = torch.stack([
            output[0]
            for output in outputs
        ]).mean(dim=0)

        r_delta_phi = torch.stack([
            output[1]
            for output in outputs
        ]).mean(dim=0)

        delta_z = torch.stack([
            output[2]
            for output in outputs
        ]).mean(dim=0)

        return delta_r, r_delta_phi, delta_z


ensemble_models = {
    side: EnsembleSpatialModel(trained_models[side]).eval()
    for side in [0, 1]
}

# Verify the trained ensembles still satisfy the exact boundary.
for side in [0, 1]:
    test_r = torch.full((200,), 53.0)
    test_phi = torch.linspace(-math.pi, math.pi, 200)
    test_z = torch.full(
        (200,),
        -z_pad_cm if side == 0 else z_pad_cm,
    )

    with torch.no_grad():
        delta_r, r_delta_phi, delta_z = ensemble_models[side](
            test_r,
            test_phi,
            test_z,
        )

    print(
        f'side {side} trained pad-plane maxima:',
        float(torch.max(torch.abs(delta_r))),
        float(torch.max(torch.abs(r_delta_phi))),
        float(torch.max(torch.abs(delta_z))),
    )


In [ ]:
# ============================================================
# Held-out residual-prediction QA
# ============================================================

residual_qa_rows = []

for side in [0, 1]:
    arrays = kshort_arrays[side]
    validation_indices = np.flatnonzero(
        arrays['event_parity'].astype(int)
        == validation_parity
    )

    flat = flatten_training_clusters(
        arrays,
        validation_indices,
    )

    model = ensemble_models[side]

    prediction_chunks = []
    chunk_size = 200000

    with torch.no_grad():
        for start in range(0, len(flat['r']), chunk_size):
            stop = min(start + chunk_size, len(flat['r']))

            r_tensor = torch.from_numpy(flat['r'][start:stop])
            phi_tensor = torch.from_numpy(flat['phi'][start:stop])
            z_tensor = torch.from_numpy(flat['z'][start:stop])

            delta_r, r_delta_phi, delta_z = model(
                r_tensor,
                phi_tensor,
                z_tensor,
            )

            predicted_normal = (
                delta_r.numpy()
                * flat['normal_projection_r'][start:stop]
                + r_delta_phi.numpy()
                * flat['normal_projection_rphi'][start:stop]
            )

            prediction_chunks.append(predicted_normal)

    predicted_normal = np.concatenate(prediction_chunks)
    target_normal = flat['res_normal']

    residual_qa_rows.append({
        'side': side,
        'clusters': len(target_normal),
        'target_mean': float(np.mean(target_normal)),
        'prediction_mean': float(np.mean(predicted_normal)),
        'rmse': float(np.sqrt(np.mean((predicted_normal - target_normal) ** 2))),
        'correlation': float(np.corrcoef(predicted_normal, target_normal)[0, 1]),
    })

    figure, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

    axes[0].hist(
        target_normal,
        bins=160,
        range=(-0.5, 0.5),
        histtype='step',
        linewidth=2,
        label='measured normal residual',
    )

    axes[0].hist(
        predicted_normal,
        bins=160,
        range=(-0.5, 0.5),
        histtype='step',
        linewidth=2,
        label='DNN conditional field',
    )

    axes[0].set_xlabel('transverse normal displacement [cm]')
    axes[0].set_ylabel('held-out clusters')
    axes[0].set_title(f'side {side}')
    axes[0].legend()

    sample_size = min(100000, len(target_normal))
    sample = np.random.default_rng(seed + side).choice(
        len(target_normal),
        size=sample_size,
        replace=False,
    )

    axes[1].hist2d(
        target_normal[sample],
        predicted_normal[sample],
        bins=120,
        range=[[-0.4, 0.4], [-0.4, 0.4]],
    )

    axes[1].plot([-0.4, 0.4], [-0.4, 0.4], linestyle='--')
    axes[1].set_xlabel('measured normal residual [cm]')
    axes[1].set_ylabel('predicted normal field [cm]')
    axes[1].set_title(f'side {side}')

    plt.show()

residual_qa_summary = pd.DataFrame(residual_qa_rows)
residual_qa_summary.to_csv(
    work_dir / 'heldout_residual_qa.csv',
    index=False,
)

display(residual_qa_summary)


In [ ]:
# ============================================================
# Held-out K0S validation, inclusive and in pT bins
# ============================================================

validation_rows = []

for side in [0, 1]:
    arrays = kshort_arrays[side]
    indices = np.flatnonzero(
        arrays['event_parity'].astype(int)
        == validation_parity
    )

    loader = DataLoader(
        CandidateDataset(arrays, indices),
        batch_size=min(validation_batch_size, len(indices)),
        shuffle=False,
    )

    metrics = evaluate_model(
        ensemble_models[side],
        loader,
        side,
        channel=0,
    )

    print(
        f'side {side} inclusive before:',
        metrics['before'],
    )
    print(
        f'side {side} inclusive after:',
        metrics['after'],
    )

    figure, axes = plt.subplots(
        2,
        3,
        figsize=(18, 10),
        constrained_layout=True,
    )

    pt_values = arrays['pair_pt'][indices]

    for bin_index, axis in enumerate(axes.flat):
        low = kshort_pt_bin_edges[bin_index]
        high = kshort_pt_bin_edges[bin_index + 1]
        select = (pt_values >= low) & (pt_values < high)

        before = metrics['mass_before'][select]
        after = metrics['mass_after'][select]

        before_metrics = peak_metrics(before)
        after_metrics = peak_metrics(after)

        validation_rows.append({
            'side': side,
            'pt_minimum': low,
            'pt_maximum': high,
            'entries': int(select.sum()),
            'mean_before': before_metrics['mean'],
            'mean_after': after_metrics['mean'],
            'width_before': before_metrics['width'],
            'width_after': after_metrics['width'],
            'width_ratio': (
                after_metrics['width'] / before_metrics['width']
                if before_metrics['width'] > 0
                else np.nan
            ),
        })

        axis.hist(
            before,
            bins=80,
            range=(0.47, 0.525),
            histtype='step',
            linewidth=2,
            label='before',
        )

        axis.hist(
            after,
            bins=80,
            range=(0.47, 0.525),
            histtype='step',
            linewidth=2,
            label='after',
        )

        axis.axvline(pdg_kshort_mass, linestyle='--')
        axis.set_title(
            f'{low:.1f} < pT(K0S) < {high:.1f}\nN={select.sum()}'
        )
        axis.set_xlabel(r'$m_{\pi\pi}$ [GeV/$c^2$]')
        axis.set_ylabel('Validation candidates')
        axis.legend()

    figure.suptitle(f'Side {side}: held-out K0S validation')
    figure.savefig(
        work_dir / f'k0s_validation_by_pt_side{side}.pdf'
    )
    plt.show()

validation_summary = pd.DataFrame(validation_rows)
validation_summary.to_csv(
    work_dir / 'k0s_validation_by_pt.csv',
    index=False,
)

display(validation_summary)


In [ ]:
# ============================================================
# Lambda and anti-Lambda validation only
# ============================================================

lambda_rows = []

for channel, label in [(1, 'Lambda'), (2, 'AntiLambda')]:
    for side in [0, 1]:
        key = (channel, side)

        if key not in validation_arrays:
            continue

        arrays = validation_arrays[key]
        indices = np.flatnonzero(
            arrays['event_parity'].astype(int)
            == validation_parity
        )

        if len(indices) == 0:
            continue

        loader = DataLoader(
            CandidateDataset(arrays, indices),
            batch_size=min(validation_batch_size, len(indices)),
            shuffle=False,
        )

        metrics = evaluate_model(
            ensemble_models[side],
            loader,
            side,
            channel=channel,
        )

        lambda_rows.append({
            'channel': label,
            'side': side,
            'entries': len(indices),
            'mean_before': metrics['before']['mean'],
            'mean_after': metrics['after']['mean'],
            'width_before': metrics['before']['width'],
            'width_after': metrics['after']['width'],
            'width_ratio': metrics['width_ratio'],
        })

        plt.figure(figsize=(8, 5))
        plt.hist(
            metrics['mass_before'],
            bins=100,
            range=(1.08, 1.16),
            histtype='step',
            linewidth=2,
            label='before',
        )
        plt.hist(
            metrics['mass_after'],
            bins=100,
            range=(1.08, 1.16),
            histtype='step',
            linewidth=2,
            label='after',
        )
        plt.axvline(1.115683, linestyle='--')
        plt.xlabel(r'$m_{p\pi}$ [GeV/$c^2$]')
        plt.ylabel('Validation candidates')
        plt.title(f'{label}, side {side}')
        plt.legend()
        plt.savefig(
            work_dir / f'{label.lower()}_validation_side{side}.pdf'
        )
        plt.show()

lambda_summary = pd.DataFrame(lambda_rows)
lambda_summary.to_csv(
    work_dir / 'lambda_validation.csv',
    index=False,
)

display(lambda_summary)


## Export the physical detector-space map

Each side histogram retains the full signed-z axis for compatibility, but the model returns exactly zero outside that side's physical domain. The QA panels display only the physical half-volume.


In [ ]:
# ============================================================
# Spatial-map QA and ROOT export
# ============================================================

r_edges = np.linspace(30.5, 76.5, 47)
phi_edges = np.linspace(-math.pi, math.pi, 49)
z_edges = np.linspace(-z_pad_cm, z_pad_cm, 43)

r_centers = 0.5 * (r_edges[:-1] + r_edges[1:])
phi_centers = 0.5 * (phi_edges[:-1] + phi_edges[1:])
z_centers = 0.5 * (z_edges[:-1] + z_edges[1:])

map_rows = []

for side in [0, 1]:
    r_grid, phi_grid, z_grid = np.meshgrid(
        r_centers,
        phi_centers,
        z_centers,
        indexing='ij',
    )

    with torch.no_grad():
        delta_r, r_delta_phi, delta_z = ensemble_models[side](
            torch.from_numpy(r_grid.astype(np.float32)),
            torch.from_numpy(phi_grid.astype(np.float32)),
            torch.from_numpy(z_grid.astype(np.float32)),
        )

    for (
        radius_value,
        phi_value,
        z_value,
        delta_r_value,
        r_delta_phi_value,
        delta_z_value,
    ) in zip(
        r_grid.ravel(),
        phi_grid.ravel(),
        z_grid.ravel(),
        delta_r.numpy().ravel(),
        r_delta_phi.numpy().ravel(),
        delta_z.numpy().ravel(),
    ):
        map_rows.append({
            'side': side,
            'r': radius_value,
            'phi': phi_value,
            'z': z_value,
            'delta_r': delta_r_value,
            'rdelta_phi': r_delta_phi_value,
            'delta_z': delta_z_value,
        })

map_frame = pd.DataFrame(map_rows)
map_frame.to_csv(
    work_dir / 'v0_spatial_residual_map_grid.csv',
    index=False,
)

# Physical phi-z slices near r=53 cm.
figure, axes = plt.subplots(
    2,
    3,
    figsize=(18, 10),
    constrained_layout=True,
)

selected_radius = r_centers[
    np.argmin(np.abs(r_centers - 53.0))
]

for side in [0, 1]:
    if side == 0:
        physical_z_selection = map_frame['z'] <= 0.0
    else:
        physical_z_selection = map_frame['z'] >= 0.0

    subset = map_frame.loc[
        (map_frame['side'] == side)
        & np.isclose(map_frame['r'], selected_radius)
        & physical_z_selection
    ]

    for column_index, (column, title) in enumerate([
        ('delta_r', r'$\Delta r$ [cm]'),
        ('rdelta_phi', r'$r\Delta\phi$ [cm]'),
        ('delta_z', r'$\Delta z$ [cm]'),
    ]):
        values = subset.pivot(
            index='z',
            columns='phi',
            values=column,
        )

        image = axes[side, column_index].pcolormesh(
            values.columns,
            values.index,
            values.values,
            shading='auto',
        )

        axes[side, column_index].set_xlabel(r'$\phi$')
        axes[side, column_index].set_ylabel('z [cm]')
        axes[side, column_index].set_title(
            f'side {side}, {title}, r~{selected_radius:.1f} cm'
        )

        figure.colorbar(
            image,
            ax=axes[side, column_index],
        )

plt.show()

# Explicit z profiles showing the hard pad-plane endpoint.
figure, axes = plt.subplots(
    2,
    3,
    figsize=(18, 9),
    constrained_layout=True,
)

profile_phi = 0.0
profile_radii = [35.0, 53.0, 70.0]

for side in [0, 1]:
    z_profile = (
        np.linspace(-z_pad_cm, 0.0, 240)
        if side == 0
        else np.linspace(0.0, z_pad_cm, 240)
    )

    for radius_value in profile_radii:
        with torch.no_grad():
            delta_r, r_delta_phi, delta_z = ensemble_models[side](
                torch.full((len(z_profile),), radius_value),
                torch.full((len(z_profile),), profile_phi),
                torch.from_numpy(z_profile.astype(np.float32)),
            )

        axes[side, 0].plot(
            z_profile,
            delta_r.numpy(),
            label=f'r={radius_value:.0f} cm',
        )

        axes[side, 1].plot(
            z_profile,
            r_delta_phi.numpy(),
            label=f'r={radius_value:.0f} cm',
        )

        axes[side, 2].plot(
            z_profile,
            delta_z.numpy(),
            label=f'r={radius_value:.0f} cm',
        )

    axes[side, 0].set_title(f'side {side}: Delta r at phi=0')
    axes[side, 1].set_title(f'side {side}: rDeltaPhi at phi=0')
    axes[side, 2].set_title(f'side {side}: Delta z at phi=0')

    for axis in axes[side]:
        axis.axhline(0.0, linestyle='--')
        axis.set_xlabel('z [cm]')
        axis.set_ylabel('correction [cm]')
        axis.legend()

plt.show()

root_file = work_dir / (
    'v0_spatial_map_with_z.root'
    if include_z_correction
    else 'v0_spatial_map_transverse_only.root'
)

if root is None:
    print('PyROOT unavailable; ROOT map not written')
else:
    output = root.TFile.Open(str(root_file), 'RECREATE')

    root.TNamed(
        'residual_convention',
        'map is measured-minus-fit field; apply corrected cluster = measured - map',
    ).Write()

    root.TNamed(
        'transverse_training_constraint',
        'normal projection: dn = delta_r*(n.er) + rdelta_phi*(n.ephi)',
    ).Write()

    root.TNamed(
        'pad_plane_boundary',
        f'exact zero at abs(z)={z_pad_cm} cm',
    ).Write()

    root.TNamed(
        'include_z_correction',
        str(int(include_z_correction)),
    ).Write()

    for side in [0, 1]:
        directory = output.mkdir(f'side{side}')
        directory.cd()

        histograms = {
            'h3_delta_r': root.TH3D(
                'h3_delta_r',
                'Delta r;r [cm];#phi;z [cm]',
                len(r_edges) - 1,
                array('d', r_edges),
                len(phi_edges) - 1,
                array('d', phi_edges),
                len(z_edges) - 1,
                array('d', z_edges),
            ),
            'h3_rdelta_phi': root.TH3D(
                'h3_rdelta_phi',
                'rDeltaPhi;r [cm];#phi;z [cm]',
                len(r_edges) - 1,
                array('d', r_edges),
                len(phi_edges) - 1,
                array('d', phi_edges),
                len(z_edges) - 1,
                array('d', z_edges),
            ),
            'h3_delta_z': root.TH3D(
                'h3_delta_z',
                'Delta z;r [cm];#phi;z [cm]',
                len(r_edges) - 1,
                array('d', r_edges),
                len(phi_edges) - 1,
                array('d', phi_edges),
                len(z_edges) - 1,
                array('d', z_edges),
            ),
        }

        subset = map_frame.loc[
            map_frame['side'] == side
        ]

        for row in subset.itertuples(index=False):
            for name, column in [
                ('h3_delta_r', 'delta_r'),
                ('h3_rdelta_phi', 'rdelta_phi'),
                ('h3_delta_z', 'delta_z'),
            ]:
                histogram = histograms[name]
                histogram.SetBinContent(
                    histogram.FindBin(row.r, row.phi, row.z),
                    getattr(row, column),
                )

        for histogram in histograms.values():
            histogram.Write()

    output.Close()
    print('Wrote:', root_file)


## Applying the map before the Kalman fit

For a cluster at \((r,\phi,z)\), look up the side-specific measured-minus-fit field and apply

\[
r_{\rm corr}=r-\Delta r,
\qquad
\phi_{\rm corr}=\phi-\frac{r\Delta\phi}{r},
\qquad
z_{\rm corr}=z-\Delta z.
\]

Then construct corrected \((x,y,z)\) measurements and run the Kalman fit. Do not apply the earlier post-fit momentum-scale or sagitta correction afterward.
